# DUK leg kinematics

Interactive inspection of the reusable `duk_core` four-bar model. This notebook calls the core API and physical geometry configuration instead of duplicating equations.


In [ ]:
from pathlib import Path
import math

from duk_core.modeling import load_leg_geometry_json, solve_leg_pose, leg_jacobian, sample_leg_trajectory
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories, plot_point_coordinates, plot_point_jacobian

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from a DUK repository checkout')

repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
geometry


In [ ]:
lower, upper = geometry.geometric_angle_limits_rad()
print(f'BO = {geometry.bo_mm:.3f} mm')
print(f'CO = {geometry.co_mm:.3f} mm')
print(f'geometric angle range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f} deg')
print('The endpoints are singular boundaries, not operating limits.')


In [ ]:
branch = 1  # Calculation branch only; verify the physical CAD branch before treating this as robot truth.
pose = solve_leg_pose(geometry, math.radians(75.0), assembly_branch=branch)
jac = leg_jacobian(geometry, pose)
print('O =', pose.o, 'mm')
print('dO/dalpha =', jac.d_o, 'mm/rad')
draw_leg_linkage(pose, geometry).set_title('DUK leg linkage — local AD frame')


In [ ]:
margin = math.radians(2.0)
poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180, assembly_branch=branch)
draw_point_trajectories(poses).set_title('B/C/H/O trajectories')


In [ ]:
plot_point_coordinates(poses, 'O').set_title('Wheel-center coordinates vs input angle')


In [ ]:
plot_point_jacobian(geometry, poses, 'O').set_title('Wheel-center Jacobian vs input angle')
